In [1]:
import pandas as pd
import numpy as np

movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

In [2]:
#Building Popularity Model
# Step 1: Count number of ratings per movie
rating_count = ratings.groupby('movieId')['rating'].count().reset_index()
rating_count.columns = ['movieId', 'rating_count']

# Step 2: Calculate average rating per movie
rating_avg = ratings.groupby('movieId')['rating'].mean().reset_index()
rating_avg.columns = ['movieId', 'avg_rating']

# Step 3: Merge both together
movie_stats = rating_count.merge(rating_avg, on='movieId')

# Step 4: Merge with movie titles
movie_stats = movie_stats.merge(movies, on='movieId')

# Step 5: Apply minimum votes threshold (only movies with 50+ ratings)
MIN_RATINGS = 50
popular_movies = movie_stats[movie_stats['rating_count'] >= MIN_RATINGS].copy()

# Step 6: Sort by average rating
popular_movies = popular_movies.sort_values('avg_rating', ascending=False)

print(f"Movies with 50+ ratings: {len(popular_movies)}")
print(popular_movies[['title', 'avg_rating', 'rating_count']].head(10))

Movies with 50+ ratings: 450
                                                  title  avg_rating  \
277                    Shawshank Redemption, The (1994)    4.429022   
659                               Godfather, The (1972)    4.289062   
2224                                  Fight Club (1999)    4.272936   
974                               Cool Hand Luke (1967)    4.271930   
602   Dr. Strangelove or: How I Learned to Stop Worr...    4.268041   
686                                  Rear Window (1954)    4.261905   
921                      Godfather: Part II, The (1974)    4.259690   
6298                               Departed, The (2006)    4.252336   
913                                   Goodfellas (1990)    4.250000   
694                                   Casablanca (1942)    4.240000   

      rating_count  
277            317  
659            192  
2224           218  
974             57  
602             97  
686             84  
921            129  
6298           107  


In [3]:
# The Weighted Rating Formula

# This is the IMDB Weighted Rating Formula
# Score = (v/(v+m)) * R + (m/(v+m)) * C
# Where:
# v = number of votes for the movie
# m = minimum votes required (our threshold)
# R = average rating of the movie
# C = mean rating across all movies

C = ratings['rating'].mean()       # mean rating across entire dataset
m = 50                              # minimum ratings threshold

def weighted_rating(x, m=m, C=C):
    v = x['rating_count']
    R = x['avg_rating']
    return (v / (v + m)) * R + (m / (v + m)) * C

popular_movies['weighted_score'] = popular_movies.apply(weighted_rating, axis=1)

# Sort by weighted score
popular_movies = popular_movies.sort_values('weighted_score', ascending=False)

print("\n🏆 Top 10 Movies by Weighted Score:")
print(popular_movies[['title', 'avg_rating', 'rating_count', 'weighted_score']].head(10))


🏆 Top 10 Movies by Weighted Score:
                                                  title  avg_rating  \
277                    Shawshank Redemption, The (1994)    4.429022   
2224                                  Fight Club (1999)    4.272936   
659                               Godfather, The (1972)    4.289062   
224           Star Wars: Episode IV - A New Hope (1977)    4.231076   
257                                 Pulp Fiction (1994)    4.197068   
46                           Usual Suspects, The (1995)    4.237745   
461                             Schindler's List (1993)    4.225000   
1938                                 Matrix, The (1999)    4.192446   
897   Star Wars: Episode V - The Empire Strikes Back...    4.215640   
314                                 Forrest Gump (1994)    4.164134   

      rating_count  weighted_score  
277            317        4.302664  
2224           218        4.129022  
659            192        4.126355  
224            251        4.109893

In [4]:
#Recommend Function
def get_popular_recommendations(genre=None, top_n=10):
    """
    Returns top N popular movies.
    Optionally filter by genre.
    """
    df = popular_movies.copy()
    
    if genre:
        df = df[df['genres'].str.contains(genre, case=False, na=False)]
    
    return df[['title', 'genres', 'avg_rating', 'rating_count', 'weighted_score']].head(top_n)

# Test it
print("=== Top 10 Overall ===")
print(get_popular_recommendations())

print("\n=== Top 5 Action Movies ===")
print(get_popular_recommendations(genre='Action', top_n=5))

=== Top 10 Overall ===
                                                  title  \
277                    Shawshank Redemption, The (1994)   
2224                                  Fight Club (1999)   
659                               Godfather, The (1972)   
224           Star Wars: Episode IV - A New Hope (1977)   
257                                 Pulp Fiction (1994)   
46                           Usual Suspects, The (1995)   
461                             Schindler's List (1993)   
1938                                 Matrix, The (1999)   
897   Star Wars: Episode V - The Empire Strikes Back...   
314                                 Forrest Gump (1994)   

                           genres  avg_rating  rating_count  weighted_score  
277                   Crime|Drama    4.429022           317        4.302664  
2224  Action|Crime|Drama|Thriller    4.272936           218        4.129022  
659                   Crime|Drama    4.289062           192        4.126355  
224       Actio

In [5]:
# Save for later use in the app
popular_movies.to_csv('../data/popular_movies.csv', index=False)
print("Saved!")

Saved!
